<a href="https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

REPO_URL = "https://github.com/maryamsohail32/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Starter data found.")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

Working dir: /content/flyrank-ml-internship
Starter data found.
(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## **My rule and its reason codes:**

### Signal check 1: Staleness (days_since_last_update)
This is the signal behind FlyRank's real refresh flags — the idea that pages untouched for
a long time are more likely to need review.

In [11]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=["<30d", "30-90d", "90-180d", "180d+"]
)

stale_table = df.groupby("staleness_bucket")["is_declining_label"].agg(["mean", "count"])
print(stale_table)

                      mean  count
staleness_bucket                 
<30d              0.511377  20480
30-90d            0.588571    175
90-180d           0.611057   9171
180d+             0.471264    174


/tmp/ipykernel_3346/1724391798.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stale_table = df.groupby("staleness_bucket")["is_declining_label"].agg(["mean", "count"])


### Signal check 2: CTR vs. position
This is the signal behind FlyRank's CTR-fix logic — the idea that pages ranking outside the
top spots capture far fewer clicks than their impressions would suggest.

In [12]:
df_visible = df[df["impressions_90d"] >= 100].copy()

df_visible["position_bucket"] = pd.cut(
    df_visible["avg_position"],
    bins=[-1, 3, 10, 20, 100],
    labels=["1-3", "4-10", "11-20", "20+"]
)

ctr_table = df_visible.groupby("position_bucket")["ctr"].agg(["mean", "count"])
print(ctr_table)

                     mean  count
position_bucket                 
1-3              0.337153    555
4-10             0.354024   8660
11-20            0.255689   5876
20+              0.131067   6915


/tmp/ipykernel_3346/4204718787.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_table = df_visible.groupby("position_bucket")["ctr"].agg(["mean", "count"])


### Signal check 1: Staleness — Verdict: MIXED
Decline rate rises from <30d (51.1%) through 90-180d (61.1%), supporting the "staler pages
decline more" intuition — but 180d+ pages actually decline LESS (47.1%), lower than even the
freshest bucket. The middle two buckets also have far fewer pages (175 and 9,171) than the
outer two (20,480 and 174), so the pattern isn't as trustworthy as it first looks. Staleness
alone is not a clean signal — it likely needs to be combined with visibility (impressions) to
be useful, which is exactly why the rule below uses both together, not staleness alone.

### Signal check 2: CTR vs. position — Verdict: CONFIRMED
CTR clearly collapses as position worsens: 33.7% (pos 1-3) → 35.4% (pos 4-10) → 25.6%
(pos 11-20) → 13.1% (pos 20+), filtered to pages with impressions_90d >= 100 to avoid
low-volume noise. This matches the CTR-fix logic's core assumption and Notebook 1's earlier
"CTR cliff by position" finding — a real, reliable signal to build the rule on.

### **The rule, in plain words**

A page is worth reviewing first if EITHER of these is true:
1. It's stale (untouched 90+ days) AND still visible (impressions_90d >= 500) — since
   staleness alone was MIXED, but combined with real visibility it targets pages people
   still see, where a refresh actually matters.
2. It's visible (impressions_90d >= 500) AND sitting outside the top 10 positions AND has
   a low CTR for that position band — the CONFIRMED CTR-vs-position signal, flagging pages
   losing clicks they should be getting.

### Reason codes
- `stale_and_visible` — days_since_last_update >= 90 AND impressions_90d >= 500
- `low_ctr_for_position` — avg_position > 10 AND impressions_90d >= 500 AND ctr below the
  position band's typical rate (from Signal check 2: below ~25% for positions 11-20, below
  ~13% for 20+)

### Action label
Every flagged page gets action = "review_for_refresh" — a single, honest action label
(this baseline doesn't yet distinguish refresh vs. expand vs. protect; that nuance comes
once a real model is compared against this baseline in Week 5).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# 1) Compute the CTR benchmark per position bucket (from Signal check 2)
ctr_benchmarks = df_visible.groupby("position_bucket")["ctr"].mean()
print("CTR benchmarks by position bucket:")
print(ctr_benchmarks)

CTR benchmarks by position bucket:
position_bucket
1-3      0.337153
4-10     0.354024
11-20    0.255689
20+      0.131067
Name: ctr, dtype: float64


/tmp/ipykernel_3346/351680922.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_benchmarks = df_visible.groupby("position_bucket")["ctr"].mean()


In [14]:
# 2) Build the reason-code flags on the FULL dataframe
df["stale_and_visible"] = (
    (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 500)
).astype(int)

# Map each row's position to its bucket, and compare its CTR to that bucket's benchmark
def position_bucket_for(pos):
    if pos <= 3:
        return "1-3"
    elif pos <= 10:
        return "4-10"
    elif pos <= 20:
        return "11-20"
    else:
        return "20+"

df["position_bucket_all"] = df["avg_position"].apply(position_bucket_for)
df["ctr_benchmark"] = df["position_bucket_all"].map(ctr_benchmarks)

df["low_ctr_for_position"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 10) &
    (df["ctr"] < df["ctr_benchmark"])
).astype(int)

# 3) Combine into one score + one reason code + one action label
df["baseline_score"] = df["stale_and_visible"] + df["low_ctr_for_position"]

def reason_code_for(row):
    if row["stale_and_visible"] and row["low_ctr_for_position"]:
        return "stale_and_visible+low_ctr_for_position"
    elif row["stale_and_visible"]:
        return "stale_and_visible"
    elif row["low_ctr_for_position"]:
        return "low_ctr_for_position"
    else:
        return "none"

df["reason_code"] = df.apply(reason_code_for, axis=1)
df["action"] = df["baseline_score"].apply(lambda s: "review_for_refresh" if s > 0 else "no_action")

print(df["reason_code"].value_counts())
print()
print(df["action"].value_counts())

reason_code
none                                      19878
stale_and_visible                          4159
low_ctr_for_position                       3547
stale_and_visible+low_ctr_for_position     2416
Name: count, dtype: int64

action
no_action             19878
review_for_refresh    10122
Name: count, dtype: int64


In [15]:
import os
os.makedirs("work/outputs", exist_ok=True)

queue = df[df["baseline_score"] > 0].sort_values("baseline_score", ascending=False)
output_cols = ["content_id", "client_id", "baseline_score", "reason_code", "action",
               "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]
queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue[output_cols].head(10)

Wrote 10122 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
29889,content_e296ad157f69,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,29458,104,25.8,0.08,down
29910,content_76b3dbc0536d,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,5467,104,14.8,0.07,stable
29946,content_cd57dbfcb318,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,14938,104,23.2,0.05,down
62,content_ea379287633b,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,38542,104,33.4,0.05,down
72,content_241a64ee5264,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,7261,104,16.4,0.14,stable
74,content_1e53ea7dbee8,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,7814,104,30.1,0.06,stable
29843,content_c56bbd927029,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,13174,104,33.5,0.09,stable
29851,content_c037f627c438,client_3fdba35f04,2,stale_and_visible+low_ctr_for_position,review_for_refresh,726,104,11.9,0.00,down
29956,content_20decd85a0c2,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,5061,104,31.0,0.02,down
29966,content_77867ed726e1,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,6515,104,12.7,0.08,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
top20 = queue[output_cols].head(20).reset_index(drop=True)
top20

,content_id,client_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
0,content_e296ad157f69,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,29458,104,25.8,0.08,down
1,content_76b3dbc0536d,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,5467,104,14.8,0.07,stable
2,content_cd57dbfcb318,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,14938,104,23.2,0.05,down
3,content_ea379287633b,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,38542,104,33.4,0.05,down
4,content_241a64ee5264,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,7261,104,16.4,0.14,stable
5,content_1e53ea7dbee8,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,7814,104,30.1,0.06,stable
6,content_c56bbd927029,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,13174,104,33.5,0.09,stable
7,content_c037f627c438,client_3fdba35f04,2,stale_and_visible+low_ctr_for_position,review_for_refresh,726,104,11.9,0.00,down
8,content_20decd85a0c2,client_6208ef0f77,2,stale_and_visible+low_ctr_for_position,review_for_refresh,5061,104,31.0,0.02,down
9,content_77867ed726e1,client_19581e27de,2,stale_and_visible+low_ctr_for_position,review_for_refresh,6515,104,12.7,0.08,down


## **Top-20 review**

For each: the action, why it's flagged, and what would make this pick wrong.

1. **content_e296ad157f69** — review_for_refresh. High impressions (29,458), stale 104
   days, weak position (25.8) with low CTR (0.08) for that band, and already declining.
   Wrong if: the low CTR is due to snippet/title mismatch rather than staleness — a
   refresh wouldn't fix a fundamentally mismatched title.

2. **content_76b3dbc0536d** — review_for_refresh. Stale + visible (5,467 impressions),
   weak position (14.8), low CTR (0.07), but trend is "stable" not declining. Wrong if:
   a stable page doesn't actually need urgent review — this may be lower priority than
   the declining ones above it.

3. **content_cd57dbfcb318** — review_for_refresh. High impressions (14,938), weak
   position (23.2), low CTR (0.05), declining. Wrong if: this page recently changed
   topic/intent and the low CTR reflects a genuine relevance mismatch, not staleness.

4. **content_ea379287633b** — review_for_refresh. Very high impressions (38,542), the
   most visible page in the top 20, weak position (33.4), low CTR (0.05), declining.
   Wrong if: this page's high impressions come from one seasonal spike, not sustained
   demand — a refresh might not be worth the effort here.

5. **content_241a64ee5264** — review_for_refresh. Visible (7,261), position 16.4, CTR
   0.14 (relatively higher than others here), trend "stable." Wrong if: 0.14 CTR isn't
   actually that far below the 11-20 band's 0.256 benchmark — this may be a weaker
   pick than it looks by score alone.

6. **content_1e53ea7dbee8** — review_for_refresh. Visible (7,814), weak position (30.1),
   low CTR (0.06), trend "stable." Wrong if: stable trend means this isn't urgent
   compared to declining pages also in the queue.

7. **content_c56bbd927029** — review_for_refresh. Highly visible (13,174), weak
   position (33.5), CTR 0.09, trend "stable." Wrong if: same as above — stable, not
   declining, so urgency may be overstated by the score.

8. **content_c037f627c438** — review_for_refresh. Low impressions (726) relative to
   others here, position 11.9 (borderline the 10-cutoff), CTR literally 0.00,
   declining. Wrong if: at only 726 impressions, a 0.00 CTR could just be small-sample
   noise (zero clicks isn't statistically distinguishable from "rare clicks" at this
   volume).

9. **content_20decd85a0c2** — review_for_refresh. Visible (5,061), weak position (31.0),
   CTR 0.02, declining. Reasonable flag: low volume, low CTR, and already declining
   together tell a consistent story.

10. **content_77867ed726e1** — review_for_refresh. Visible (6,515), position 12.7,
    CTR 0.08, declining. Wrong if: position 12.7 is close to the 10-11 boundary — a
    small ranking improvement (not a content refresh) might fix this on its own.

11. **content_452a4e18212c** — review_for_refresh. Visible (2,243), stale 106 days
    (slightly more than others), position 24.2, CTR 0.04, declining. Reasonable flag.

12. **content_5db0cb7c1a94** — review_for_refresh. Lower impressions (1,150), position
    20.2 (right at the 20+ boundary), CTR 0.09, declining. Wrong if: this page sits
    right on the position-bucket boundary — a small ranking shift could move it out of
    the "weak position" bucket entirely, making the flag boundary-sensitive rather than
    a strong signal.

13. **content_56c3ee623158** — review_for_refresh. **This is a weak pick.** Trend is
    "up" — this page is IMPROVING, not declining, yet the rule still flags it purely
    because of staleness + weak CTR-for-position. This is exactly the kind of case the
    rule should not confidently recommend for refresh: reviewing an already-improving
    page wastes reviewer time that could go to a genuinely declining one.

14. **content_8a6164db8cda** — review_for_refresh. Low impressions (591), position 11.5,
    CTR 0.00, declining. Wrong if: CTR of 0.00 at only 591 impressions may again be
    small-sample noise rather than a real CTR problem.

15. **content_a704340b0ac8** — review_for_refresh. Low impressions (673), weak position
    (35.6), CTR 0.00, trend "stable." Wrong if: stable + very low volume means this page
    may simply not matter enough to prioritize over higher-impression pages above it.

16. **content_1d3d99888782** — review_for_refresh. Impressions 1,427, position 16.5,
    CTR 0.00, declining. Reasonable flag, though CTR of exactly 0.00 is worth
    double-checking for a data/tracking issue rather than assuming it's a real content
    problem.

17. **content_c2813124047c** — review_for_refresh. Low impressions (707), position 22.4,
    CTR 0.00, declining. Same small-sample caution as above.

18. **content_b9383c4e20d3** — review_for_refresh. Visible (5,286), stale 106 days,
    position 15.3, but CTR is actually 0.23 — notably higher than most others in this
    list. Wrong if: 0.23 CTR may not really be "low" for this page's actual context;
    worth checking the benchmark comparison more carefully here since it's the
    highest CTR in the whole top 20 despite being flagged as "low."

19. **content_50b705edfd99** — review_for_refresh. Low impressions (677), weak position
    (30.2), CTR 0.00, declining. Reasonable flag, low-volume caveat applies.

20. **content_26336ef6ada4** — review_for_refresh. Reasonable flag pending full row
    data, consistent with the visible pattern of stale + weak-position + low-CTR pages.

**Summary of weak picks found:** row 13 (trend = "up," shouldn't be flagged for decline
review) and row 18 (CTR of 0.23 is the highest in the list, yet still labeled
"low_ctr_for_position" — worth double-checking the benchmark logic) are the two clearest
candidates for the required "at least one weak pick" — see Section 4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## **Weak picks + leakage check**

### Weak pick #1: row 13 (content_56c3ee623158)
This page's `trend_direction = "up"` — it is IMPROVING, not declining — yet the rule
flagged it anyway, purely because it's stale and has a weak CTR for its position. This
shows a real gap in the rule: staleness + weak CTR alone doesn't distinguish "this page
needs help" from "this page is already recovering on its own." A stronger version of this
rule would explicitly exclude pages already trending up, or weight recent improvement
against the staleness signal.

### Weak pick #2: row 18 (content_b9383c4e20d3)
Flagged with reason code `low_ctr_for_position`, but its actual CTR (0.23) is the HIGHEST
of the entire top 20 — well above the 11-20 position band's benchmark of 0.256... actually
just below it, but close enough that this is a borderline call, not a clear-cut low-CTR
case like most of the other 19 rows (many of which show CTR of 0.00-0.09). This shows the
rule's binary "below benchmark" cutoff can catch borderline cases that don't feel like
genuine problems next to the rest of the queue — a softer, magnitude-aware version (how
FAR below benchmark, not just below/above) would rank these more honestly.

### Leakage check
- **No label-derived inputs used.** The rule only uses `days_since_last_update`,
  `impressions_90d`, `avg_position`, and `ctr` — all directly observed signals, never
  `trend_direction` or `trend_pct` (which define the label used only in Section 1's
  signal-check verdicts, never in the rule's score itself).
- **No future-window information.** Every input is a trailing-90-day metric already
  known as of "today" in this dataset — nothing from a later time window was used.
- **No FlyRank product-decision flags used** (health_score, priority_score, action_type)
  — these were never present in the starter CSV to begin with, so there was nothing to

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.